In [ ]:
# This script preprocesses the patch history variables up to a decision point before visiting the next patch.
# The output file ForGLM.csv is for comparing the patch chosen to be revisited vs the average values of the patches not
# visited. The output file ForGLM_NewVsRev.csv looks at agent history variables up to a decision point before the 
# visiting the next patch, but comparing variables before going to a new patch vs revisiting an old one. This NewVsRev GLM is not 
# used because of statistics issues and is still under development.

import csv
import pandas as pd
import numpy as np
import copy
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import glob
import math

pd.options.mode.chained_assignment = None  # default='warn'

In [ ]:
startCSV1 = 0
startCSV2 = 0
maxAgent = 5 # number of runs/agents
MaxSession = []
EpDF = {}
#EpDF["EpisodeLists"] = np.empty((len(df), 0)).tolist()
glued_data = pd.DataFrame()
sessionID2 = 0

for agentID in range(maxAgent):
    ep_ids = []
    agentIDst = str(agentID)
    directoryPath0 = '/Baseline/' 
    directoryPath = directoryPath0 + agentIDst +'O/' 
    sessionID = 0

    for file_name in glob.glob(directoryPath+'*.csv'):
        x = pd.read_csv(file_name, low_memory=False)
        Ldata = len(x['id'])
        #print(Ldata)
        if Ldata > 2000:
            x['agentID'] = agentID
            x['sessionID'] = sessionID
            x['xerror'] = x['pred_delta_x'] - x['delta_x'] 
            x['yerror'] = x['pred_delta_y'] - x['delta_y'] 
            x['uncertainty'] = np.sqrt(x['xerror']*x['xerror'] + x['yerror']*x['yerror'])             
            sessionID = sessionID + 1
            sessionID2 = sessionID2 + 1
            glued_data = pd.concat([glued_data,x],axis=0)

    subIDdf = glued_data[glued_data['agentID'] == agentID]
    ep_ids = np.unique(subIDdf.id.values)
    EpDF[f"EpIDs_{agentID}"] = ep_ids
    print(ep_ids)
    MaxSession.append(len(ep_ids))

    
for agentID in range(maxAgent):
    for sessionID in range( len(EpDF[f"EpIDs_{agentID}"]) ):
        print(agentID)
        df2 = glued_data[glued_data['id'] == EpDF[f"EpIDs_{agentID}"][sessionID]]
        df = df2[df2['agentID'] == agentID]
        df["PatchList"] = np.empty((len(df), 0)).tolist()
        df["PL_Recency"] = np.empty((len(df), 0)).tolist()
        df["PL_Distance"] = np.empty((len(df), 0)).tolist()
        df["PL_PredatorRate"] = np.empty((len(df), 0)).tolist()
        df["PL_FoodRate"] = np.empty((len(df), 0)).tolist()
        df["PL_DrinkRate"] = np.empty((len(df), 0)).tolist()
        df["PL_HungerLvl"] = np.empty((len(df), 0)).tolist()
        df["PL_ThirstLvl"] = np.empty((len(df), 0)).tolist()
        df["PL_EnergyLvl"] = np.empty((len(df), 0)).tolist()
        df["PL_RPE"] = np.empty((len(df), 0)).tolist()
        df["TotalDuration"] = np.empty((len(df), 0)).tolist()
        df["PL_Cows"] = np.empty((len(df), 0)).tolist()
        df['PL_Uncertainty'] = np.empty((len(df), 0)).tolist()
        df["PL_NumPredKilled"] = np.empty((len(df), 0)).tolist()
        df["PL_HasSword"] = np.empty((len(df), 0)).tolist()
        df["PL_HasPick"] = np.empty((len(df), 0)).tolist()
        df["PL_HasIron"] = np.empty((len(df), 0)).tolist()
        
        df["has_pick"][df["has_pick"] > 0] = 1
        df["has_sword"][df["has_sword"] > 0] = 1


        maxQuadNum = int( df['quadrant'].max() ) + 1
        Xcoords = np.ones(maxQuadNum)*-1
        Ycoords = np.ones(maxQuadNum)*-1
        lastQuadIdx = np.ones((len(df),maxQuadNum))*-1

        temp = {}
        temp['vec'] = np.empty((len(df), 0)).tolist()
        for i in range(len(df['is_revisit_patch'])):
            temp['vec'][i].append(-1)

        PatchListVec = []
        for i in range(len(df['is_revisit_patch'])):
            if df.iloc[i]['dist_to_melee_l1'] < 8:
                df.iloc[i]['isRunPred'] = 1    

            quad = int(df.iloc[i]['quadrant'])
            Xcoords[quad] = df.iloc[i]["player_position_x"]
            Ycoords[quad] = df.iloc[i]["player_position_y"]
            
            if(df.iloc[i]['is_new_patch']==1):
                if (df.iloc[i]['quadrant'] in PatchListVec) == False :
                    PatchListVec.append(df.iloc[i]['quadrant'])
                    #print(PatchListVec)
            df.at[i,'PatchList'] = copy.deepcopy(PatchListVec)
            #temp['vec'][i] = copy.deepcopy(PatchListVec)
            #if i < 100:
                #print( df['PatchList'][i] )
            if i > 0:
                lastQuadIdx[i,:] = lastQuadIdx[i-1,:]
                lastQuadIdx[i,quad] = i

        t0X = 0
        t0Y = 0
        t0cat = 0
        t0predRate = 0
        t0NumCows = 0

        t1X = 0
        t1Y = 0
        t1cat = 0

        NNvec = []
        RNvec = []
        NRvec = []
        RRvec = []
        NNvecPred = []
        RNvecPred = []
        NRvecPred = []
        RRvecPred = []
        NNvecCow = []
        RNvecCow = []
        NRvecCow = []
        RRvecCow = []

        window0 = 100

        for i in range (len(df['is_revisit_patch'])): 
            #if i%500 == 0:
                #print(i)
            if i > window0:    
                if (df['revisit_patch_eat_drink'][i] == 1 and df['revisit_patch_eat_drink'][i-1] == 0) or (df['new_patch_eat_drink'][i] == 1 and df['new_patch_eat_drink'][i-1] == 0):
                    if t0X == 0 and t0Y == 0:
                        t0X = df['player_position_x'][i]
                        t0Y = df['player_position_y'][i]
                        if (df['revisit_patch_eat_drink'][i] == 1 and df['revisit_patch_eat_drink'][i-1] == 0):
                            t0cat = 1
                        if (df['new_patch_eat_drink'][i] == 1 and df['new_patch_eat_drink'][i-1] == 0):
                            t0cat = 2
                        df_subset0 = df[(i-window0):i]    
                        t0predRate = np.sum(df_subset0["num_melee_nearby"]) + np.sum(df_subset0["num_ranged_nearby"])
                        t0NumCows = np.sum(df_subset0["num_passives_nearby"])

                    else:
                        t1X = df['player_position_x'][i]
                        t1Y = df['player_position_y'][i]
                        if (df['revisit_patch_eat_drink'][i] == 1 and df['revisit_patch_eat_drink'][i-1] == 0):
                            t1cat = 1
                        if (df['new_patch_eat_drink'][i] == 1 and df['new_patch_eat_drink'][i-1] == 0):
                            t1cat = 2
                        InterPatchDist = np.sqrt( (t1X-t0X)*(t1X-t0X) + (t1Y-t0Y)*(t1Y-t0Y) )


                        if t1cat == 2 and t0cat == 2:
                            NNvec.append(InterPatchDist)
                            NNvecPred.append(t0predRate)
                            NNvecCow.append(t0NumCows)
                        if t1cat == 2 and t0cat == 1:
                            RNvec.append(InterPatchDist)
                            RNvecPred.append(t0predRate)
                            RNvecCow.append(t0NumCows)
                        if t1cat == 1 and t0cat == 2:
                            NRvec.append(InterPatchDist)
                            NRvecPred.append(t0predRate)
                            NRvecCow.append(t0NumCows)
                        if t1cat == 1 and t0cat == 1:
                            RRvec.append(InterPatchDist)
                            RRvecPred.append(t0predRate)
                            RRvecCow.append(t0NumCows)

                        t0X = copy.deepcopy(t1X)
                        t0Y = copy.deepcopy(t1X)
                        t0cat = copy.deepcopy(t1cat)
                        df_subset0 = df[i-window0:i]
                        t0predRate = np.sum(df_subset0["num_melee_nearby"]) + np.sum(df_subset0["num_ranged_nearby"])
                        t0NumCows = np.sum(df_subset0["num_passives_nearby"])


            for j in df['PatchList'][i]:
                dist = np.sqrt( (df['player_position_x'][i]-Xcoords[j])*(df['player_position_x'][i]-Xcoords[j]) + (df['player_position_y'][i]-Ycoords[j])*(df['player_position_y'][i]-Ycoords[j]) )
                df["PL_Distance"][i].append(int(dist))    
                timeRecency = int ( i-lastQuadIdx[i,j])
                df["PL_Recency"][i].append(timeRecency)

                df_subset = df[0:i+1]
                df["PL_FoodRate"][i].append(np.sum(df_subset["is_eat"][df_subset["quadrant"]==j])/len(df_subset["is_eat"][df_subset["quadrant"]==j]))
                df["PL_DrinkRate"][i].append(np.sum(df_subset["is_drink"][df_subset["quadrant"]==j])/len(df_subset["is_drink"][df_subset["quadrant"]==j]))
                df["PL_PredatorRate"][i].append(np.sum(df_subset["num_melee_nearby"][df_subset["quadrant"]==j])/len(df_subset["num_melee_nearby"][df_subset["quadrant"]==j]) + np.sum(df_subset["num_ranged_nearby"][df_subset["quadrant"]==j])/len(df_subset["num_ranged_nearby"][df_subset["quadrant"]==j]))  
                df["PL_HungerLvl"][i].append(np.sum(df_subset["food"][df_subset["quadrant"]==j])/len(df_subset["food"][df_subset["quadrant"]==j]))
                df["PL_ThirstLvl"][i].append(np.sum(df_subset["drink"][df_subset["quadrant"]==j])/len(df_subset["drink"][df_subset["quadrant"]==j]))
                df["PL_EnergyLvl"][i].append(np.sum(df_subset["energy"][df_subset["quadrant"]==j])/len(df_subset["energy"][df_subset["quadrant"]==j]))
                df["TotalDuration"][i].append(len(df_subset["energy"][df_subset["quadrant"]==j]))  
                df["PL_Cows"][i].append(np.sum(df_subset["num_passives_nearby"][df_subset["quadrant"]==j])/len(df_subset["num_passives_nearby"][df_subset["quadrant"]==j])) 
                df['PL_Uncertainty'][i].append(np.sum(df_subset["uncertainty"][df_subset["quadrant"]==j])/len(df_subset["uncertainty"][df_subset["quadrant"]==j]))
                df['PL_NumPredKilled'][i].append(np.sum(df_subset["num_monsters_killed"][df_subset["quadrant"]==j])/len(df_subset["num_monsters_killed"][df_subset["quadrant"]==j]))
                df['PL_HasPick'][i].append(np.sum(df_subset["has_pick"][df_subset["quadrant"]==j])/len(df_subset["has_pick"][df_subset["quadrant"]==j]))
                df['PL_HasSword'][i].append(np.sum(df_subset["has_sword"][df_subset["quadrant"]==j])/len(df_subset["has_sword"][df_subset["quadrant"]==j]))
                  
          

                
        window = 100
        HungerPrev = 0
        ThirstPrev = 0
        EnergyPrev = 0
        EatRatePrev = 0
        DrinkRatePrev = 0
        MeleePrev = 0
        UncertaintyPrev = 0
        PredKillRatePrev = 0
        HasSwordRatePrev = 0
        HasPickRatePrev = 0


        for i in range (len(df['revisit_patch_eat_drink'])): 
           # if i%500 == 0:
              #  print(i)
            if i > window:    
                if (df['revisit_patch_eat_drink'][i] == 1 and df['revisit_patch_eat_drink'][i-1] == 0) or (df['new_patch_eat_drink'][i] == 1 and df['new_patch_eat_drink'][i-1] == 0):
                    if (df['revisit_patch_eat_drink'][i] == 1 and df['revisit_patch_eat_drink'][i-1] == 0):
                            HungerPrev = np.mean(df['food'][(i-window):i])
                            ThirstPrev = np.mean(df['drink'][(i-window):i])
                            EnergyPrev = np.mean(df['energy'][(i-window):i])
                            EatRatePrev = np.mean(df['is_eat'][(i-window):i])
                            DrinkRatePrev = np.mean(df['is_drink'][(i-window):i])
                            MeleePrev = np.mean(df['num_melee_nearby'][(i-window):i]) + np.mean(df['num_ranged_nearby'][(i-window):i])
                            CowsClose = np.mean(df['num_passives_nearby'][(i-window):i]) 
                            UncertaintyPrev = np.mean(df['uncertainty'][(i-window):i]) 
                            PredKillRatePrev = np.mean(df['num_monsters_killed'][(i-window):i])
                            HasSwordRatePrev = np.mean(df['has_sword'][(i-window):i])
                            HasPickRatePrev = np.mean(df['has_pick'][(i-window):i])

                            if startCSV1 == 1:  
                                filename1 = directoryPath0 + 'ForGLM_NewVsRev.csv'
                                file = open(filename1, 'a+', newline ='') 
                                with file:
                                    # identifying header  
                                    header = ['AgentID','SessionID','SessionID2','PatchNewOrRevisit', 'EatRate', 'DrinkRate', 'PredRate', 'Hunger', 'Thirst', 'Energy', 'CowCount', 'Uncertainty',
                                             'PredKillRate','HasSwordRate','HasPickRate']
                                    writer = csv.DictWriter(file, fieldnames = header)             
                                    writer.writerow({'AgentID' : agentID,'SessionID' : sessionID,'SessionID2' : sessionID2,'PatchNewOrRevisit' : 0, 'EatRate' : EatRatePrev, 
                                                     'DrinkRate' : DrinkRatePrev, 'PredRate' : MeleePrev, 'Hunger' : HungerPrev, 
                                                     'Thirst' : ThirstPrev, 'Energy' : EnergyPrev, 'CowCount' : CowsClose,'Uncertainty' : UncertaintyPrev,
                                                     'PredKillRate' : PredKillRatePrev,'HasSwordRate' : HasSwordRatePrev,'HasPickRate' : HasPickRatePrev})
                                    #writer.writerow({'PatchRevisit' : 0, 'Food' : food_OQ/m, 'Drink' : drink_OQ/m, 'PredRate' : pred_OQ/m, 'Hunger' : hunger_OQ/m, 'Thirst' : thirst_OQ/m, 'Energy' : energy_OQ/m, 'Distance' : dist_OQ/m, 'Recency' : recency_OQ/m, 'Dwelltime' : dur_OQ/m})

                            if startCSV1 == 0:                
                                # writing data row-wise into the csv file
                                filename1 = directoryPath0 + 'ForGLM_NewVsRev.csv'
                                file = open(filename1, 'w', newline ='') 
                                with file:
                                    header = ['AgentID','SessionID','SessionID2','PatchNewOrRevisit', 'EatRate', 'DrinkRate', 'PredRate', 'Hunger', 'Thirst', 'Energy', 'CowCount''Uncertainty',
                                             'PredKillRate','HasSwordRate','HasPickRate']
                                    writer = csv.DictWriter(file, fieldnames = header)     
                                    writer.writeheader()
                                    startCSV1 = 1

                    if (df['new_patch_eat_drink'][i] == 1 and df['new_patch_eat_drink'][i-1] == 0):
                            HungerPrev = np.mean(df['food'][(i-window):i])
                            ThirstPrev = np.mean(df['drink'][(i-window):i])
                            EnergyPrev = np.mean(df['energy'][(i-window):i])
                            EatRatePrev = np.mean(df['is_eat'][(i-window):i])
                            DrinkRatePrev = np.mean(df['is_drink'][(i-window):i])
                            MeleePrev = np.mean(df['num_melee_nearby'][(i-window):i]) + np.mean(df['num_ranged_nearby'][(i-window):i])
                            CowsClose = np.mean(df['num_passives_nearby'][(i-window):i]) 
                            UncertaintyPrev = np.mean(df['uncertainty'][(i-window):i]) 
                            PredKillRatePrev = np.mean(df['num_monsters_killed'][(i-window):i])
                            HasSwordRatePrev = np.mean(df['has_sword'][(i-window):i])
                            HasPickRatePrev = np.mean(df['has_pick'][(i-window):i])

                            if startCSV1 == 1:    
                                filename1 = directoryPath0 + 'ForGLM_NewVsRev.csv'
                                file = open(filename1, 'a+', newline ='') 
                                with file:
                                    # identifying header  
                                    header = ['AgentID','SessionID','SessionID2','PatchNewOrRevisit', 'EatRate', 'DrinkRate', 'PredRate', 'Hunger', 'Thirst', 'Energy','CowCount', 'Uncertainty',
                                             'PredKillRate','HasSwordRate','HasPickRate']
                                    writer = csv.DictWriter(file, fieldnames = header)             
                                    writer.writerow({'AgentID' : agentID,'SessionID' : sessionID,'SessionID2' : sessionID2,'PatchNewOrRevisit' : 1, 'EatRate' : EatRatePrev, 
                                                     'DrinkRate' : DrinkRatePrev, 'PredRate' : MeleePrev, 'Hunger' : HungerPrev, 'Thirst' : ThirstPrev, 
                                                     'Energy' : EnergyPrev, 'CowCount' : CowsClose,'Uncertainty' : UncertaintyPrev,
                                                     'PredKillRate' : PredKillRatePrev,'HasSwordRate' : HasSwordRatePrev,'HasPickRate' : HasPickRatePrev})
                                    #writer.writerow({'PatchRevisit' : 0, 'Food' : food_OQ/m, 'Drink' : drink_OQ/m, 'PredRate' : pred_OQ/m, 'Hunger' : hunger_OQ/m, 'Thirst' : thirst_OQ/m, 'Energy' : energy_OQ/m, 'Distance' : dist_OQ/m, 'Recency' : recency_OQ/m, 'Dwelltime' : dur_OQ/m})

                            if startCSV1 == 0:                
                                # writing data row-wise into the csv file
                                filename1 = directoryPath0 + 'ForGLM_NewVsRev.csv'
                                file = open(filename1, 'w', newline ='') 
                                with file:
                                    header = ['AgentID','SessionID','SessionID2','PatchNewOrRevisit', 'EatRate', 'DrinkRate', 'PredRate', 'Hunger', 'Thirst', 
                                              'Energy', 'CowCount', 'Uncertainty','PredKillRate','HasSwordRate','HasPickRate']
                                    writer = csv.DictWriter(file, fieldnames = header)     
                                    writer.writeheader()
                                    startCSV1 = 1

        foodFact = []
        drinkFact = []
        predFact = []
        hungerFact = []
        thirstFact = []
        energyFact = []
        distFact = []
        recencyFact = []
        durFact = []
        cowFact = []

        summaryMean = []
        summarySD = []
        timeBeforePatchVisit = 50


        for idx2 in range (len(df['is_revisit_patch'])):
            if idx2 > 1000:
                #if (df['QuadNextRevisit'][i] > -1) and (df['QuadNextRevisit'][i] != df['QuadNextRevisit'][i-1]):
                if ( (df['is_eat'][idx2] > 0) and df['is_revisit_patch'][idx2] == 1):
                    NextQuad = df['quadrant'][idx2]   
                    
                    i = idx2 - timeBeforePatchVisit
                    
                    vecIdx = 0
                    food_NQ = 0
                    drink_NQ = 0
                    pred_NQ = 0
                    hunger_NQ = 0
                    thirst_NQ = 0
                    energy_NQ = 0
                    dist_NQ = 0
                    recency_NQ = 0
                    dur_NQ = 0
                    cows_NQ = 0
                    unc_NQ = 0
                    predKills_NQ = 0
                    hasPick_NQ = 0
                    hasSword_NQ = 0

                    food_OQ = 0
                    drink_OQ = 0
                    pred_OQ = 0
                    hunger_OQ = 0
                    thirst_OQ = 0
                    energy_OQ = 0
                    count_OQ = 0
                    dist_OQ = 0
                    recency_OQ = 0
                    dur_OQ = 0
                    cows_OQ = 0
                    unc_OQ = 0
                    predKills_OQ = 0
                    hasPick_OQ = 0
                    hasSword_OQ = 0

                    for j in df['PatchList'][i]:
                        if j == NextQuad:
                            food_NQ = df["PL_FoodRate"][i][vecIdx]
                            drink_NQ = df["PL_DrinkRate"][i][vecIdx]
                            pred_NQ = df["PL_PredatorRate"][i][vecIdx] 
                            hunger_NQ = df["PL_HungerLvl"][i][vecIdx]
                            thirst_NQ = df["PL_ThirstLvl"][i][vecIdx]
                            energy_NQ = df["PL_EnergyLvl"][i][vecIdx]
                            dist_NQ = df["PL_Distance"][i][vecIdx]
                            recency_NQ = df["PL_Recency"][i][vecIdx]
                            dur_NQ = df["TotalDuration"][i][vecIdx]
                            cows_NQ = df["PL_Cows"][i][vecIdx]
                            unc_NQ = df["PL_Uncertainty"][i][vecIdx]
                            predKills_NQ = df['PL_NumPredKilled'][i][vecIdx]
                            hasPick_NQ = df['PL_HasPick'][i][vecIdx]
                            hasSword_NQ = df['PL_HasSword'][i][vecIdx]                           
                         

                        if j != NextQuad:
                            count_OQ = count_OQ + 1
                            food_OQ = df["PL_FoodRate"][i][vecIdx] + food_OQ
                            drink_OQ = df["PL_DrinkRate"][i][vecIdx] + drink_OQ
                            pred_OQ = df["PL_PredatorRate"][i][vecIdx] + pred_OQ
                            hunger_OQ = df["PL_HungerLvl"][i][vecIdx] + hunger_OQ
                            thirst_OQ = df["PL_ThirstLvl"][i][vecIdx] + thirst_OQ
                            energy_OQ = df["PL_EnergyLvl"][i][vecIdx] + energy_OQ
                            dist_OQ = df["PL_Distance"][i][vecIdx] + dist_OQ
                            recency_OQ = df["PL_Recency"][i][vecIdx] + recency_OQ
                            dur_OQ = df["TotalDuration"][i][vecIdx] + dur_OQ
                            cows_OQ = df["PL_Cows"][i][vecIdx] + cows_OQ
                            unc_OQ = df["PL_Uncertainty"][i][vecIdx] + unc_OQ
                            predKills_OQ = df['PL_NumPredKilled'][i][vecIdx] + predKills_OQ
                            hasPick_OQ = df['PL_HasPick'][i][vecIdx] + hasPick_OQ
                            hasSword_OQ = df['PL_HasSword'][i][vecIdx] + hasSword_OQ

                        vecIdx = vecIdx + 1


                    if startCSV2 == 1:    
                        m = count_OQ
                        if m > 0:
                            filename2 = directoryPath0 + 'ForGLM.csv'
                            file = open(filename2, 'a+', newline ='') 
                            print(df['PatchList'][i])
                            with file:
                                # identifying header  
                                header = ['AgentID','SessionID','SessionID2','PatchRevisit', 'EatRate', 'DrinkRate', 'PredRate', 'Hunger', 
                                          'Thirst', 'Energy', 'Distance', 'Recency', 'Dwelltime', 'CowCount', 'Uncertainty','nPatches',
                                         'PredKillRate','HasSwordRate','HasPickRate']
                                writer = csv.DictWriter(file, fieldnames = header)             
                                writer.writerow({'AgentID' : agentID,'SessionID' : sessionID,'SessionID2' : sessionID2,'PatchRevisit' : 1, 'EatRate' : food_NQ, 
                                                 'DrinkRate' : drink_NQ, 'PredRate' : pred_NQ, 'Hunger' : hunger_NQ, 'Thirst' : thirst_NQ, 
                                                 'Energy' : energy_NQ, 'Distance' : dist_NQ, 'Recency' : recency_NQ, 'Dwelltime' : dur_NQ, 
                                                 'CowCount' : cows_NQ, 'Uncertainty' : unc_NQ, 'nPatches': len(df['PatchList'][i]),
                                                 'PredKillRate' : predKills_NQ, 'HasSwordRate' : hasSword_NQ,'HasPickRate' : hasPick_NQ})
                                writer.writerow({'AgentID' : agentID,'SessionID' : sessionID,'SessionID2' : sessionID2,'PatchRevisit' : 0, 'EatRate' : food_OQ/m, 
                                                 'DrinkRate' : drink_OQ/m, 'PredRate' : pred_OQ/m, 'Hunger' : hunger_OQ/m, 
                                                 'Thirst' : thirst_OQ/m, 'Energy' : energy_OQ/m, 'Distance' : dist_OQ/m, 'Recency' : recency_OQ/m, 
                                                 'Dwelltime' : dur_OQ/m, 'CowCount' : cows_OQ/m, 'Uncertainty' : unc_OQ/m, 'nPatches': len(df['PatchList'][i]),
                                                 'PredKillRate' : predKills_OQ, 'HasSwordRate' : hasSword_OQ,'HasPickRate' : hasPick_OQ})


                                #if math.isnan(unc_OQ/m) == True:
                                    #print(m)
                                    #print(unc_OQ)
                                    #print(df['PatchList'][i])
                                    #print(sessionID)
                                    #print(agentID)
                                    #print(i)
                                    #print(j)
                                    #print(df["PL_FoodRate"][i])
                                    #printthirst_OQ()
                                
                    if startCSV2 == 0:                
                            # writing data row-wise into the csv file
                            filename2 = directoryPath0 + 'ForGLM.csv'
                            file = open(filename2, 'w', newline ='') 
                            with file:
                                header = ['AgentID','SessionID','SessionID2','PatchRevisit', 'EatRate', 'DrinkRate', 'PredRate', 'Hunger', 
                                          'Thirst', 'Energy', 'Distance', 'Recency', 'Dwelltime', 'CowCount', 'Uncertainty','nPatches',
                                          'PredKillRate','HasSwordRate','HasPickRate']
                                writer = csv.DictWriter(file, fieldnames = header)     
                                writer.writeheader()
                                startCSV2 = 1    

        summaryMean = [np.nanmean(foodFact),np.nanmean(drinkFact),np.nanmean(predFact),np.nanmean(hungerFact),np.nanmean(thirstFact),np.nanmean(energyFact),np.nanmean(distFact),np.nanmean(recencyFact),np.nanmean(durFact),np.nanmean(cowFact)]
        summarySD = [np.nanstd(foodFact),np.nanstd(drinkFact),np.nanstd(predFact),np.nanstd(hungerFact),np.nanstd(thirstFact),np.nanstd(energyFact),np.nanstd(distFact),np.nanstd(recencyFact),np.nanstd(durFact),np.nanstd(cowFact)] 
        summarySEM = [np.nanstd(foodFact)/np.sqrt(len(foodFact)),np.nanstd(drinkFact)/np.sqrt(len(drinkFact)),np.nanstd(predFact)/np.sqrt(len(predFact)),np.nanstd(hungerFact)/np.sqrt(len(hungerFact)),np.nanstd(thirstFact)/np.sqrt(len(thirstFact)),np.nanstd(energyFact)/np.sqrt(len(energyFact)),np.nanstd(distFact)/np.sqrt(len(distFact)),np.nanstd(recencyFact)/np.sqrt(len(recencyFact)),np.nanstd(durFact)/np.sqrt(len(durFact)),np.nanstd(cowFact)/np.sqrt(len(cowFact))] 

